# 02 Python Unit Tests - unittest - zaawansowane
_Kamil Bartocha_ | _wersja 2.0_

## Rozkład jazdy

1. 🔹 **`setUp` / `tearDown` i cykl zycia testu** - lifecycle, `setUpClass`
2. 🔹 **`@skip`, `@skipIf`, `@expectedFailure`** - pomijanie testow
3. 🔹 **`subTest` - parametryzacja w unittest**

🐍 Każda sekcja zawiera ćwiczenia.

## 1. 🔹 `setUp` / `tearDown` i cykl zycia testu

Klasa `TestCase` definiuje cztery metody sterujace cyklem zycia
testow. Pozwalaja one przygotowac srodowisko testowe i posprzatac
po wykonaniu testow:

| Metoda | Kiedy wywolywana | Typowe uzycie |
|--------|-----------------|---------------|
| `setUpClass(cls)` | raz przed wszystkimi testami klasy | kosztowna inicjalizacja (baza, serwer) |
| `setUp(self)` | przed kazdym testem | swiezy obiekt CUT |
| `tearDown(self)` | po kazdym tescie (nawet gdy fail) | sprzatanie zasobow |
| `tearDownClass(cls)` | raz po wszystkich testach klasy | zamkniecie bazy, serwera |

Kolejnosc wywolan dla dwoch testow w klasie:

```
setUpClass
  setUp -> test_a -> tearDown
  setUp -> test_b -> tearDown
tearDownClass
```

`setUpClass` i `tearDownClass` musza byc dekorowane
`@classmethod` i przyjmowac `cls` zamiast `self`.

> 💡 **Tip:** `tearDown` jest wywolywane nawet gdy test failuje.
> Dzieki temu zasoby (pliki, polaczenia) sa zawsze zwalniane.
> Uzyj `setUp` do tworzenia swiezego obiektu CUT przed kazdym
> testem - eliminuje efekty uboczne miedzy testami.

In [ ]:
import unittest


class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount


class TestBankAccountLifecycle(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        print("\n[setUpClass] - raz przed wszystkimi testami")
        cls.initial_balance = 100

    @classmethod
    def tearDownClass(cls):
        print("[tearDownClass] - raz po wszystkich testach")

    def setUp(self):
        # Swiezy obiekt przed kazdym testem - brak efektow ubocznych
        print(f"  [setUp] przed: {self._testMethodName}")
        self.account = BankAccount(balance=self.initial_balance)

    def tearDown(self):
        print(f"  [tearDown] po: {self._testMethodName}")

    def test_deposit_increases_balance(self):
        self.account.deposit(50)
        self.assertEqual(self.account.balance, 150)

    def test_withdraw_decreases_balance(self):
        self.account.withdraw(30)
        self.assertEqual(self.account.balance, 70)

    def test_initial_balance_is_correct(self):
        self.assertEqual(self.account.balance, 100)


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(
    TestBankAccountLifecycle
))

---

### 🐍 Ćwiczenia - `setUp` / `tearDown` i cykl zycia

**Ćw. 1.** Napisz klase `TestBankAccount` uzywajac `setUp`
do tworzenia swiezego obiektu `BankAccount` przed kazdym testem.
Dodaj minimum 3 metody testowe.

**Ćw. 2.** Napisz klase `TestUserRepository` uzywajac
`setUpClass` do jednorazowego stworzenia bazy SQLite w pamieci
i `tearDownClass` do jej zamkniecia. Testy wykonuja zapytania
na tej wspolnej bazie.

**Ćw. 3. *(Trudniejsze)*** Napisz klase `TestFileStorage`
uzywajac `setUp` do tworzenia tymczasowego pliku (`tempfile`)
i `tearDown` do jego usuniecia. Przetestuj zapis i odczyt.

In [ ]:
# Ćw. 1: setUp tworzacy swiezy obiekt BankAccount
import unittest


class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount


class TestBankAccount(unittest.TestCase):

    def setUp(self):
        # hint: stworz swiezy obiekt BankAccount z konkretnym saldem
        ...

    def test_initial_balance(self):
        ...

    def test_deposit_increases_balance(self):
        ...

    def test_withdraw_decreases_balance(self):
        ...


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestBankAccount))

In [ ]:
# Ćw. 2: setUpClass z baza SQLite - operacja kosztowna
import unittest
import sqlite3


class TestUserRepository(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        # hint: stworz polaczenie SQLite :memory: i tabele users
        #       zapisz polaczenie jako cls.conn
        ...

    @classmethod
    def tearDownClass(cls):
        # hint: zamknij polaczenie cls.conn
        ...

    def test_insert_user_and_count(self):
        # hint: wstaw uzytkownika i sprawdz COUNT(*)
        ...

    def test_fetch_user_by_name(self):
        # hint: wstaw i pobierz uzytkownika po imieniu
        ...


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestUserRepository))

In [ ]:
# Ćw. 3: setUp/tearDown z plikiem tymczasowym
import unittest
import tempfile
import os


class FileStorage:
    def save(self, path, content):
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()


class TestFileStorage(unittest.TestCase):

    def setUp(self):
        # hint: tempfile.NamedTemporaryFile(delete=False, suffix=".txt")
        #       zapisz sciezke jako self.tmp_path i self.storage
        ...

    def tearDown(self):
        # hint: usun plik os.unlink(self.tmp_path) jezeli istnieje
        ...

    def test_save_and_load_returns_original_content(self):
        ...

    def test_save_overwrites_existing_content(self):
        ...


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestFileStorage))

## 2. 🔹 `@skip`, `@skipIf`, `@expectedFailure`

Modul `unittest` dostarcza dekoratorow pozwalajacych pomijac testy
lub oznaczyc je jako spodziewanie bledne:

| Dekorator | Zachowanie |
|-----------|------------|
| `@unittest.skip(reason)` | zawsze pomija test |
| `@unittest.skipIf(cond, reason)` | pomija gdy `cond` jest `True` |
| `@unittest.skipUnless(cond, reason)` | pomija gdy `cond` jest `False` |
| `@unittest.expectedFailure` | test MUSI failowac - inaczej blad |

Dekoratory mozna stosowac do metod testowych lub do calej klasy.

```python
@unittest.skipIf(sys.version_info < (3, 10), "Wymaga Python 3.10+")
def test_match_statement(self):
    ...

@unittest.expectedFailure
def test_known_bug(self):
    self.assertEqual(broken_function(), 42)  # wiadomo ze failuje
```

Wyniki w raporcie:
- `s` - test pominiêty (skipped)
- `x` - spodziewana porazka (expected failure) - traktowana jako OK
- `u` - nieoczekiwany sukces (unexpected success) - traktowany jako blad

> 💡 **Tip:** `@expectedFailure` dokumentuje znany bug, ktory
> nie zostal jeszcze naprawiony. Gdy naprawisz buga, test
> zacznie przechodzic - dostaniesz `unexpected success`
> jako przypomnienie, ze nalezy usunac dekorator.

In [ ]:
import sys
import unittest


def greet(name):
    return f"Hello, {name}!"


def parse_version(version_str):
    """Parsuje napis '3.10.1' na tuple (3, 10, 1)."""
    return tuple(int(x) for x in version_str.split("."))


class TestSkipDecorators(unittest.TestCase):

    @unittest.skip("Funkcja w trakcie implementacji - WIP")
    def test_greet_with_title(self):
        self.assertEqual(greet("Dr. Smith"), "Hello, Dr. Smith!")

    @unittest.skipIf(sys.version_info < (3, 10), "Wymaga Python >= 3.10")
    def test_greet_basic(self):
        self.assertEqual(greet("Alice"), "Hello, Alice!")

    @unittest.skipUnless(sys.platform == "linux", "Tylko na Linux")
    def test_platform_specific(self):
        self.assertIn("linux", sys.platform)

    @unittest.expectedFailure
    def test_parse_version_known_bug(self):
        # Ten test dokumentuje znany blad - parse_version nie
        # obsługuje sufiksow takich jak '3.10.1b2'
        result = parse_version("3.10.1b2")
        self.assertEqual(result, (3, 10, 1))


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestSkipDecorators))

---

### 🐍 Ćwiczenia - `@skip`, `@skipIf`, `@expectedFailure`

**Ćw. 4.** Pomiń test dla Pythona < 3.10 uzywajac
`@skipIf(sys.version_info < (3, 10), ...)`. Dodaj rowniez
test z `@skipUnless` sprawdzajacy system operacyjny.

**Ćw. 5.** Oznacz test jako `@expectedFailure` dla funkcji
z celowym bledem. Sprawdz wynik - powinien byc `x` (xfail).
Nastepnie napraw funkcje i sprawdz, co sie zmieni.

**Ćw. 8.** Polaczy klasy `TestBankAccount` (Ćw. 1) i
`TestFileStorage` (Ćw. 3) w `TestSuite` i uruchom razem.

In [ ]:
# Ćw. 4: @skipIf i @skipUnless
import sys
import unittest


def format_error_code(code):
    """Formatuje kod bledu jako napis np. 'ERR-404'."""
    return f"ERR-{code}"


class TestSkipConditions(unittest.TestCase):

    @unittest.skipIf(sys.version_info < (3, 10), "Wymaga Python >= 3.10")
    def test_format_error_code_basic(self):
        self.assertEqual(format_error_code(404), "ERR-404")

    @unittest.skipUnless(..., "Tylko na Windows")
    def test_windows_specific_behavior(self):
        # hint: uzyj sys.platform == "win32" jako warunku
        ...

    @unittest.skipIf(..., "Pomijamy w trybie CI")
    def test_skipped_in_ci(self):
        # hint: sprawdz zmienna srodowiskowa os.environ.get("CI")
        import os
        ...


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestSkipConditions))

In [ ]:
# Ćw. 5: @expectedFailure - znany blad
import unittest


# Funkcja z celowym bledem (nie obsluguje wartosci ujemnych)
def absolute_value(n):
    if n > 0:
        return n
    return 0  # BUG: powinno byc -n dla n < 0


class TestAbsoluteValue(unittest.TestCase):

    def test_absolute_value_positive(self):
        self.assertEqual(absolute_value(5), 5)

    @unittest.expectedFailure
    def test_absolute_value_negative_known_bug(self):
        # hint: ten test musi failowac (wynik to 0, powinno byc 3)
        ...

    # Po naprawieniu bledu powyzszy test da "unexpected success".
    # Odkomentuj ponizej po naprawieniu funkcji:
    # def test_absolute_value_negative_fixed(self):
    #     self.assertEqual(absolute_value(-3), 3)


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestAbsoluteValue))

In [ ]:
# Ćw. 8: TestSuite laczacy dwie klasy testowe
import unittest


class BankAccount:
    def __init__(self, balance=0): self.balance = balance
    def deposit(self, amount):     self.balance += amount
    def withdraw(self, amount):    self.balance -= amount


class TextUtils:
    def upper(self, text): return text.upper()
    def reverse(self, text): return text[::-1]


class TestBankAccount(unittest.TestCase):
    def setUp(self):
        self.account = BankAccount(balance=100)

    def test_deposit(self):
        self.account.deposit(50)
        self.assertEqual(self.account.balance, 150)

    def test_withdraw(self):
        self.account.withdraw(30)
        self.assertEqual(self.account.balance, 70)


class TestTextUtils(unittest.TestCase):
    def setUp(self):
        self.utils = TextUtils()

    def test_upper(self):
        self.assertEqual(self.utils.upper("hello"), "HELLO")

    def test_reverse(self):
        self.assertEqual(self.utils.reverse("abc"), "cba")


# Polaczy obie klasy w TestSuite:
suite = ...

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 3. 🔹 `subTest` - parametryzacja w unittest

`subTest` to context manager pozwalajacy uruchamiac ten sam
test wielokrotnie z roznymi danymi wejsciowymi. Kluczowa
roznica w stosunku do petli `for`:

- bez `subTest`: pierwszy fail zatrzymuje caly test,
- z `subTest`: kazda iteracja jest niezalezna - raportowane
  sa wszystkie niepowodzenia naraz.

```python
def test_is_even_for_multiple_inputs(self):
    cases = [0, 2, 4, 6, 8]
    for n in cases:
        with self.subTest(n=n):   # etykieta w raporcie bledu
            self.assertTrue(is_even(n))
```

Parametry przekazane do `subTest()` pojawiaja sie w komunikacie
błedu i pomagaja identyfikowac ktory przypadek zawiodl:

```
FAIL: test_is_even (n=3)
```

Mozna tez przekazac wiele etykiet:
```python
with self.subTest(input=text, expected=expected):
    self.assertEqual(slugify(text), expected)
```

> 💡 **Tip:** `subTest` jest swietny do testowania funkcji
> czystych (pure functions) z wieloma parami wejscie/wyjscie.
> Dla bardziej zlozonych scenariuszy z innymi frameworkami
> uzyj `@pytest.mark.parametrize` (modul 05).

In [ ]:
import unittest


def classify_age(age):
    """Klasyfikuje wiek jako 'minor', 'adult' lub 'senior'."""
    if age < 0:
        raise ValueError("Age cannot be negative")
    if age < 18:
        return "minor"
    if age < 65:
        return "adult"
    return "senior"


class TestClassifyAge(unittest.TestCase):

    def test_classify_age_for_multiple_cases(self):
        cases = [
            (0,   "minor"),
            (17,  "minor"),
            (18,  "adult"),
            (30,  "adult"),
            (64,  "adult"),
            (65,  "senior"),
            (100, "senior"),
        ]
        for age, expected in cases:
            with self.subTest(age=age, expected=expected):
                self.assertEqual(classify_age(age), expected)

    def test_classify_age_boundary_values(self):
        # subTest z jednym parametrem jako etykieta
        boundaries = {17: "minor", 18: "adult", 64: "adult", 65: "senior"}
        for age, expected in boundaries.items():
            with self.subTest(age=age):
                self.assertEqual(classify_age(age), expected)


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestClassifyAge))

---

### 🐍 Ćwiczenia - `subTest`

**Ćw. 6. *(Trudniejsze)*** Uzyj `subTest` do przetestowania
funkcji `is_valid_age()` dla co najmniej 10 wartosci:
zarówno prawidlowych, jak i nieprawidlowych (w tym granicznych).

**Ćw. 7.** Napisz `subTest` dla listy par wejscie/oczekiwany
wynik dla funkcji `slugify(text)` (zamiana tekstu na slug URL).

**Ćw. 9. *(Trudniejsze)*** Napisz bazowa klase `BaseAccountTest`
z `setUp` tworzacym obiekt `BankAccount`. Nastepnie dziedzicz
po niej w klasach `TestDeposit` i `TestWithdraw`.

In [ ]:
# Ćw. 6: subTest dla is_valid_age() - co najmniej 10 wartosci
import unittest


def is_valid_age(age):
    """Zwraca True jezeli wiek jest liczba calkowita z zakresu [0, 150]."""
    return isinstance(age, int) and 0 <= age <= 150


class TestIsValidAge(unittest.TestCase):

    def test_valid_ages(self):
        # hint: lista wartosci True: 0, 1, 18, 65, 150
        valid = [...]
        for age in valid:
            with self.subTest(age=age):
                ...

    def test_invalid_ages(self):
        # hint: lista wartosci False: -1, 151, 3.5, None, "18", True
        # hint: isinstance(True, int) == True, wiec True nie jest validnym wiekiem
        invalid = [...]
        for age in invalid:
            with self.subTest(age=age):
                ...


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestIsValidAge))

In [ ]:
# Ćw. 7: subTest dla slugify()
import unittest


def slugify(text):
    """Zamienia tekst na slug URL: lowercase, spacje na '-',
    usuwa znaki specjalne."""
    import re
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s-]", "", text)
    text = re.sub(r"[\s-]+", "-", text)
    return text.strip("-")


class TestSlugify(unittest.TestCase):

    def test_slugify_cases(self):
        cases = [
            # (wejscie,               oczekiwany slug)
            ("Hello World",           "hello-world"),
            ("  spaces  ",            ...),
            ("multiple   spaces",     ...),
            ("special!@#chars",       ...),
            ("already-a-slug",        ...),
            # hint: dodaj co najmniej 2 wlasne przypadki
            ...,
            ...,
        ]
        for text, expected in cases:
            with self.subTest(text=text, expected=expected):
                self.assertEqual(slugify(text), expected)


runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestSlugify))

In [ ]:
# Ćw. 9: bazowa klasa TestCase z setUp - dziedziczenie
import unittest


class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount


# Klasa bazowa z wspolnym setUp
class BaseAccountTest(unittest.TestCase):
    def setUp(self):
        # hint: stworz self.account = BankAccount(balance=200)
        ...


# Klasa testowa dla depositu - dziedziczy setUp z BaseAccountTest
class TestDeposit(BaseAccountTest):

    def test_deposit_increases_balance(self):
        ...

    def test_deposit_negative_raises(self):
        ...


# Klasa testowa dla wyplaty - dziedziczy setUp z BaseAccountTest
class TestWithdraw(BaseAccountTest):

    def test_withdraw_decreases_balance(self):
        ...

    def test_withdraw_exceeds_balance_raises(self):
        ...


# Uruchom obie klasy
suite = unittest.TestSuite()
suite.addTests(unittest.TestLoader().loadTestsFromTestCase(TestDeposit))
suite.addTests(unittest.TestLoader().loadTestsFromTestCase(TestWithdraw))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)